<a href="https://colab.research.google.com/github/aldenbarnes/structure_properties_fall_2026/blob/main/Code/address_comps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Address → Rental Comps

Run the cells from top to bottom.

1. **Type an address** in Step 1.
2. **Step 2** looks the address up in our San Francisco property file (`Data/properties.csv`) to get its
   coordinates and `analysis_neighborhood`, then finds its **zip code**.
3. **Step 3** asks RealtyAPI for rental listings. It makes **exactly two requests**:
   * up to 500 listings from **all of San Francisco**
   * up to 500 listings from the address's **zip code**
4. **Step 4** saves the results as CSV files.

RealtyAPI charges credits per request, so the notebook stops itself after two requests. Re-running Step 3
will **not** send new requests; restart the runtime if you really want to pull again.

## Step 0: Setup
Add your RealtyAPI key to Colab once: click the **🔑 key icon** in the left sidebar, add a secret named
`REALTYAPI_KEY`, and turn on notebook access. If you skip this, Step 3 will ask you to paste the key.

In [ ]:
import json
import re
from datetime import datetime

import pandas as pd
import requests
from pathlib import Path

pd.set_option("display.max_columns", 50)

## Step 1: Enter the address
Street number and street name are enough. Unit numbers, city, state and zip are fine too; they get cleaned off.

`ZIP_CODE` is optional. Leave it as `None` and Step 2 finds it for you.

In [ ]:
ADDRESS = "1010 Anza St"   # <-- change me
ZIP_CODE = None            # optional, e.g. "94118" to skip the zip lookup

## Step 2: Look up the address in our San Francisco property file
This searches `Data/properties.csv` (built by `download_SF.ipynb` from the SF Assessor and permit data).
Every row is one residential address, with its `analysis_neighborhood`, coordinates and building details.

In [ ]:
PROPERTIES_PATH = Path("Data/properties.csv")
PROPERTIES_URL = "https://raw.githubusercontent.com/aldenbarnes/structure_properties_fall_2026/main/Data/properties.csv"

# Use the repo copy if it's been cloned, otherwise download it from GitHub
properties = pd.read_csv(PROPERTIES_PATH if PROPERTIES_PATH.exists() else PROPERTIES_URL,
                         dtype={"parcel_number": str})
print(f"Loaded {len(properties):,} San Francisco properties")

### Clean up the address so it matches the file's format
The file writes addresses like `1010 ANZA ST`, so we upper-case the input, drop the city/state/zip,
drop any unit number, and shorten words like `STREET` → `ST`.

In [ ]:
STREET_TYPES = {
    "STREET": "ST", "AVENUE": "AVE", "BOULEVARD": "BLVD", "DRIVE": "DR", "ROAD": "RD",
    "PLACE": "PL", "COURT": "CT", "LANE": "LN", "TERRACE": "TER", "WAY": "WAY",
    "ALLEY": "ALY", "CIRCLE": "CIR", "HIGHWAY": "HWY", "PLAZA": "PLZ", "PARKWAY": "PKWY",
}


def clean_address(text):
    """'1010 Anza Street, Apt 2, San Francisco, CA 94118' -> '1010 ANZA ST'"""
    text = text.upper().split(",")[0]                  # keep only the street part
    text = re.sub(r"\s+(APT|UNIT|STE|SUITE|#).*$", "", text)  # drop unit numbers
    text = re.sub(r"[^A-Z0-9 ]", " ", text)             # drop punctuation
    words = [STREET_TYPES.get(w, w) for w in text.split()]
    return " ".join(words)


search_address = clean_address(ADDRESS)
print("Searching for:", search_address)

### Find the matching row
`address_unit` is the street address, plus a unit number for condos (e.g. `100 MAIN ST 4`).
We look for an exact match first, then for unit rows at that address.

In [ ]:
file_address = properties["address_unit"].fillna("").str.upper()

matches = properties[file_address == search_address]
if matches.empty:
    matches = properties[file_address.str.startswith(search_address + " ")]

if matches.empty:
    similar = properties[file_address.str.contains(search_address.split(" ", 1)[-1], regex=False)]
    print("Addresses on that street:", similar["address_unit"].head(15).tolist())
    raise ValueError(f"'{search_address}' is not in properties.csv. Check the spelling in Step 1.")

print(f"{len(matches)} matching row(s)")
matches

### Find the zip code
`properties.csv` has no zip column, so we send the property's coordinates to the U.S. Census geocoder
(free, no key, not RealtyAPI) and read back which zip code area it falls in.

In [ ]:
subject = matches.iloc[0]


def zip_from_coordinates(latitude, longitude):
    response = requests.get(
        "https://geocoding.geo.census.gov/geocoder/geographies/coordinates",
        params={"x": longitude, "y": latitude, "benchmark": "Public_AR_Current",
                "vintage": "Current_Current", "layers": "all", "format": "json"},
        timeout=60,
    )
    response.raise_for_status()
    geographies = response.json()["result"]["geographies"]
    for layer, areas in geographies.items():
        if "ZIP" in layer.upper() and areas:
            area = areas[0]
            return str(area.get("ZCTA5") or area.get("BASENAME") or area.get("GEOID"))[:5]
    return None


zip_code = ZIP_CODE
if not zip_code:
    typed_zip = re.search(r"\b(94\d{3})\b", ADDRESS)   # a zip typed into ADDRESS
    zip_code = typed_zip.group(1) if typed_zip else zip_from_coordinates(subject["latitude"], subject["longitude"])

if not zip_code:
    raise ValueError("Could not find the zip code. Set ZIP_CODE in Step 1, e.g. ZIP_CODE = '94118'.")
print("Zip code:", zip_code)

### The address's geography

In [ ]:
property_info = {
    "address": subject["address_unit"],
    "zip_code": zip_code,
    "analysis_neighborhood": subject["analysis_neighborhood"],
    "latitude": subject["latitude"],
    "longitude": subject["longitude"],
    "parcel_number": subject["parcel_number"],
    "zoning_code": subject["zoning_code"],
    "year_built": subject["year_built"],
    "units": subject["units"],
    "building_sqft": subject["building_sqft"],
    "rows_matched": len(matches),
}

pd.Series(property_info, name="Your property").to_frame()

## Step 3: Pull rental listings from RealtyAPI (2 requests total)
Uses the same `/search/bylocation` endpoint as `realtyapi_notebook.ipynb`. 500 is the most one request can
return.

In [ ]:
REALTYAPI_URL = "https://apartments.realtyapi.io/search/bylocation"
RESULTS_PER_REQUEST = 500
MAX_REQUESTS = 2

# Load the key from Colab secrets, or ask for it
try:
    from google.colab import userdata
    REALTYAPI_KEY = userdata.get("REALTYAPI_KEY")
except Exception:
    import getpass
    REALTYAPI_KEY = getpass.getpass("RealtyAPI key (hidden): ").strip()

# Keeps track of requests so re-running cells can't spend extra credits
if "api_responses" not in globals():
    api_responses = {}


def search_listings(label, location):
    """Ask RealtyAPI for up to 500 listings in `location`. Each label is only ever requested once."""
    if label in api_responses:
        print(f"[{label}] already downloaded, reusing it (no new request).")
        return api_responses[label]
    if len(api_responses) >= MAX_REQUESTS:
        raise RuntimeError(f"Stopped: already made {MAX_REQUESTS} requests this session.")

    print(f"[{label}] requesting up to {RESULTS_PER_REQUEST} listings for '{location}'...")
    response = requests.get(
        REALTYAPI_URL,
        params={"location": location, "page": 1, "resultCount": RESULTS_PER_REQUEST},
        headers={"x-realtyapi-key": REALTYAPI_KEY, "Accept": "application/json"},
        timeout=120,
    )
    if not response.ok:
        # Failed requests still count toward the cap, in case they cost credits
        api_responses[label] = {"error": f"HTTP {response.status_code}"}
        raise RuntimeError(f"[{label}] RealtyAPI returned HTTP {response.status_code}. Check your key and credits.")
    api_responses[label] = response.json()

    data = api_responses[label]
    print(f"[{label}] got {len(data.get('searchResults', []))} listings "
          f"(RealtyAPI reports {data.get('total')} total matches)")
    return data

### Request 1 of 2: all of San Francisco

In [ ]:
sf_response = search_listings("san_francisco", "San Francisco, CA")

### Request 2 of 2: the property's zip code

In [ ]:
zip_response = search_listings("zip_code", f"San Francisco, CA {zip_code}")

### Turn the responses into tables
Each row is one listing (a building or unit for rent). Nested fields are flattened into columns like
`address.postalCode`.

In [ ]:
def to_table(response, search_name):
    df = pd.json_normalize(response.get("searchResults", []))
    df.insert(0, "search", search_name)
    return df


sf_listings = to_table(sf_response, "San Francisco")
zip_listings = to_table(zip_response, f"Zip {zip_code}")

print(f"San Francisco listings: {len(sf_listings)}")
print(f"Zip {zip_code} listings:  {len(zip_listings)}")

PREVIEW_COLUMNS = ["search", "name", "oneLineAddress", "address.postalCode",
                   "propertyType", "bedRange", "rentRange", "latitude", "longitude", "listingKey"]
zip_listings[[c for c in PREVIEW_COLUMNS if c in zip_listings]].head(20)

In [ ]:
sf_listings[[c for c in PREVIEW_COLUMNS if c in sf_listings]].head(20)

## Step 4: Save the results
Files go in `realtyapi_output/` with the address and date in the name. In Colab, open the 📁 folder icon on
the left to download them. The raw JSON is saved too, so nothing is lost and nothing needs re-requesting.

In [ ]:
output_dir = Path("realtyapi_output")
output_dir.mkdir(exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M")
slug = search_address.replace(" ", "_")

all_listings = pd.concat([sf_listings, zip_listings], ignore_index=True)
all_listings["subject_address"] = property_info["address"]
all_listings["subject_zip"] = zip_code

all_listings.to_csv(output_dir / f"{slug}_{stamp}_listings.csv", index=False)
pd.Series(property_info).to_csv(output_dir / f"{slug}_{stamp}_property.csv", header=["value"])
(output_dir / f"{slug}_{stamp}_raw.json").write_text(json.dumps(api_responses, indent=2))

print("Saved:")
for f in sorted(output_dir.glob(f"{slug}_{stamp}_*")):
    print("   ", f)
print(f"Requests used this session: {len(api_responses)} of {MAX_REQUESTS}")